# Global Getis-Ord G: test overall concentration

This lab calculates global Getis-Ord G for positive values on a fixed neighbour graph. We will inspect the observed statistic, its expectation and a permutation reference distribution, then change the values without changing the graph.

**You need:** Python, NumPy, Matplotlib, libpysal and esda.

Global G is a whole-study-area statistic. It does not tell us where hotspots occur; local Gi* does that.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from libpysal.weights import W
from esda import G


## 1. Create a binary neighbour graph

The graph is a ring of twelve units. Every unit is connected to the two units beside it. The weights are binary and the diagonal is zero.

In [ ]:
n = 12
neighbors = {i: [(i - 1) % n, (i + 1) % n] for i in range(n)}
w = W(neighbors)
w.transform = 'B'
values = np.array([10, 11, 9, 2, 1, 2, 8, 9, 10, 1, 2, 1], dtype=float)

np.random.seed(12345)
global_g = G(values, w, permutations=999)

print('Observed G:', global_g.G)
print('Expected G:', global_g.EG)
print('Analytical z-score:', global_g.z_norm)
print('Analytical p-value:', global_g.p_norm)
print('Permutation p-value:', global_g.p_sim)


In [ ]:
angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
xy = np.column_stack([np.cos(angles), np.sin(angles)])
fig, ax = plt.subplots(figsize=(6, 6))
for i, linked in neighbors.items():
    for j in linked:
        if i < j:
            ax.plot([xy[i, 0], xy[j, 0]], [xy[i, 1], xy[j, 1]], linewidth=2)
ax.scatter(xy[:, 0], xy[:, 1], s=850, zorder=3)
for i, (x, y) in enumerate(xy):
    ax.text(x, y, f'U{i}\n{values[i]:g}', ha='center', va='center', color='white', weight='bold')
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Synthetic ring graph: positive values')
plt.show()


## 2. Inspect the permutation reference distribution

The permutation procedure keeps the neighbour graph fixed and rearranges the observed values across the units. The observed G is compared with the simulated values.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(global_g.sim, bins=20)
ax.axvline(global_g.G, linewidth=2, label=f'Observed G = {global_g.G:.4f}')
ax.axvline(global_g.EG, linestyle='--', linewidth=2, label=f'Expected G = {global_g.EG:.4f}')
ax.set_xlabel('Global G across permutations')
ax.set_ylabel('Frequency')
ax.set_title('Permutation reference distribution')
ax.legend()
plt.show()


## 3. Change the values, keep the graph

Now make the values more even. The graph does not change. Compare the observed G, expected value and z-score with the previous result.

In [ ]:
even_values = np.array([5, 6, 5, 4, 5, 4, 5, 6, 5, 4, 5, 4], dtype=float)
np.random.seed(12345)
global_g_even = G(even_values, w, permutations=999)
print('Even values:', even_values.tolist())
print('Observed G:', global_g_even.G)
print('Expected G:', global_g_even.EG)
print('Analytical z-score:', global_g_even.z_norm)
print('Permutation p-value:', global_g_even.p_sim)


## What to take away

- Global G tests for an overall concentration of positive values.
- It uses raw values and binary spatial weights in the standard PySAL workflow.
- The expected value depends on the sum of weights and the number of units.
- A global result does not locate the hotspot; use local Gi* for that.
- The direction of the z-score matters when interpreting high versus low concentration.

Continue with [Getis-Ord Gi*](../algorithms/statistics/getis-ord-gi-star.md) and the [global spatial autocorrelation learning path](../learning-paths/global-spatial-autocorrelation.md).

**Reference:** [PySAL's Global Getis-Ord G guide](https://pysal.org/esda/stable/user-guide/getisord.html) and [API reference](https://pysal.org/esda/stable/generated/esda.G.html).